# 02 — Wrangling e feature engineering temporale

## Obiettivi didattici

1. Distinguere wrangling "deterministico" (ok prima dello split) da imputazione statistica (deve stare dentro la pipeline).
2. Generare feature temporali (rolling, diff, z-score) per asset.
3. Verificare che le feature derivate non leakino il futuro.
4. Eseguire lo split temporale (no shuffle!) sui primi 7 giorni come training.


In [1]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from iot_anomaly.data import load_raw, time_split
from iot_anomaly.wrangling import add_missing_flags, fill_missing_per_asset
from iot_anomaly.features import TimeSeriesFeatureEngineer, select_modeling_features
from iot_anomaly.config import DEFAULT_CONFIG, SENSOR_COLUMNS


In [2]:
df = load_raw()
cols = SENSOR_COLUMNS + ('ambient_temp_c', 'humidity_pct', 'load_pct')
df = add_missing_flags(df, columns=cols)
df = fill_missing_per_asset(df, columns=cols)
fe = TimeSeriesFeatureEngineer(window=15)
df_fe = fe.fit_transform(df)
print(f'Colonne: {df.shape[1]} → {df_fe.shape[1]}')
print(f'Feature derivate: {df_fe.shape[1] - df.shape[1]}')


Colonne: 30 → 62
Feature derivate: 32


## Verifica anti-leakage delle rolling

Una rolling che usa un campione del **futuro** sarebbe leakage. La nostra implementazione usa `pandas.Series.rolling(window).mean()` che è back-looking: al tempo *t* aggrega `[t-w+1, t]`, mai `[t, t+w-1]`. Verifichiamolo manualmente.

In [3]:
asset0 = df_fe[df_fe.asset_id == 0].iloc[:25][['timestamp','vib_rms','vib_rms_roll_mean']]
asset0.head(20)

,timestamp,vib_rms,vib_rms_roll_mean
0,2025-02-01 00:00:00+00:00,0.017158,0.017158
1,2025-02-01 00:01:00+00:00,0.012610,0.014884
2,2025-02-01 00:02:00+00:00,0.012175,0.013981
3,2025-02-01 00:03:00+00:00,0.015814,0.014439
4,2025-02-01 00:04:00+00:00,0.007602,0.013072
5,2025-02-01 00:05:00+00:00,0.007182,0.012090
6,2025-02-01 00:06:00+00:00,0.197512,0.038579
7,2025-02-01 00:07:00+00:00,0.014293,0.035543
8,2025-02-01 00:08:00+00:00,0.016783,0.033459
9,2025-02-01 00:09:00+00:00,0.239781,0.054091


## Time-aware split

Splittiamo il dataset cronologicamente: i primi 7 giorni → training (considerati "normali"), gli ultimi 3 giorni → test. Tutti i 16 asset sono presenti in entrambi i set; cambia solo la finestra temporale.

In [4]:
train, test, cutoff = time_split(df_fe, DEFAULT_CONFIG)
print(f'Cutoff: {cutoff}')
print(f'train: {len(train):,} righe, test: {len(test):,} righe')
print(f'asset in train: {train.asset_id.nunique()}, in test: {test.asset_id.nunique()}')
print(f'anomaly_label train: {train.anomaly_label.mean():.2%}, test: {test.anomaly_label.mean():.2%}')


Cutoff: 2025-02-08 00:00:00+00:00
train: 161,280 righe, test: 69,120 righe
asset in train: 16, in test: 16
anomaly_label train: 1.92%, test: 3.42%


## Feature di modellazione finali

Le feature usate dal clustering combinano:
- Sensori grezzi
- Rolling mean/std/diff/zscore
- Variabili di contesto (load_pct, humidity, ambient_temp)


In [5]:
modeling = select_modeling_features(df_fe)
print(f'{len(modeling)} feature di modellazione:')
for c in modeling:
    print(f'  - {c}')


43 feature di modellazione:
  - rpm
  - current_a
  - pressure_bar
  - flow_lpm
  - temp_c
  - vib_rms
  - vib_crest
  - vib_kurtosis
  - rpm_roll_mean
  - rpm_roll_std
  - current_a_roll_mean
  - current_a_roll_std
  - pressure_bar_roll_mean
  - pressure_bar_roll_std
  - flow_lpm_roll_mean
  - flow_lpm_roll_std
  - temp_c_roll_mean
  - temp_c_roll_std
  - vib_rms_roll_mean
  - vib_rms_roll_std
  - vib_crest_roll_mean
  - vib_crest_roll_std
  - vib_kurtosis_roll_mean
  - vib_kurtosis_roll_std
  - rpm_diff
  - current_a_diff
  - pressure_bar_diff
  - flow_lpm_diff
  - temp_c_diff
  - vib_rms_diff
  - vib_crest_diff
  - vib_kurtosis_diff
  - rpm_zscore
  - current_a_zscore
  - pressure_bar_zscore
  - flow_lpm_zscore
  - temp_c_zscore
  - vib_rms_zscore
  - vib_crest_zscore
  - vib_kurtosis_zscore
  - ambient_temp_c
  - humidity_pct
  - load_pct
